# Streamlit Demo (Colab, T4 GPU, tunneled)

**Project**: Cybersecurity Domain RAG Pipeline — Final Year Project (CB013309)
**Scope**: run the existing `src/app.py` Streamlit demo (baseline Groq + fine-tuned Phi-2
side-by-side comparison) on a Colab GPU, exposed through a temporary public URL — no local
CPU inference needed.

This does not change `app.py` at all — it reuses it as-is, which already imports the fixed
`query_finetuned()` from `src/rag_pipeline.py` (GPU device placement, token-budget fix).

**Requires a GPU runtime**: `Runtime > Change runtime type > T4 GPU`.

**Ephemeral by design**: the demo dies when this Colab runtime disconnects, and the public
URL changes every time you run the tunnel cell. This is meant for a live walkthrough /
one-off demo, not a permanent deployment. To stop the demo, interrupt the tunnel cell
(or the runtime) — this also tears down the background Streamlit process.

In [ ]:
!pip uninstall -y torchao -q

!pip install -q \
    streamlit \
    plotly \
    markdown \
    pyngrok \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    langchain \
    langchain-classic \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-experimental \
    langchain-huggingface \
    langchain-groq \
    chromadb \
    sentence-transformers \
    python-dotenv

import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected — go to Runtime > Change runtime type > T4 GPU.")

## Step 1 — Mount Google Drive and configure paths

- `ADAPTER_DIR_DRIVE` — if present, its contents are copied over the repo's bundled
  `models/lora_adapter` before launching (same precedence as the evaluation notebook). If it
  isn't there, the adapter already committed in the repo is used as-is.
- `GROQ_API_KEY` — needed for the baseline side. Read from Colab's **Secrets** manager (key
  icon in the left sidebar — add a secret named `GROQ_API_KEY` once, and it's available in
  every future session) if present, otherwise you'll be prompted for it (hidden input) below.
  Never printed, never committed either way.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# ── Configure these to match your Drive layout ─────────────────────────────
ADAPTER_DIR_DRIVE = "/content/drive/MyDrive/FYP/output/lora_adapter"
REPO_GIT_URL      = "https://github.com/ZuhriAshroff/cybersecurity-rag-llm-fyp-v2.git"
REPO_DIR          = "/content/fyp_v2"
STREAMLIT_PORT    = 8501
# ────────────────────────────────────────────────────────────────────────────

print("Adapter (Drive):", ADAPTER_DIR_DRIVE, "exists:", os.path.isdir(ADAPTER_DIR_DRIVE))

## Step 2 — Sync the repo and set the Groq API key

Clones (or pulls) the project repo, copies the Drive adapter over the bundled one if present,
and writes your Groq key into a `.env` file inside the cloned repo (`src/rag_pipeline.py`
loads it via `python-dotenv`, exactly as it does locally).

In [ ]:
import subprocess, shutil, os
from getpass import getpass

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, REPO_DIR], check=True)

repo_adapter_dir = os.path.join(REPO_DIR, "models", "lora_adapter")
if os.path.isdir(ADAPTER_DIR_DRIVE):
    print(f"Copying adapter from Drive ({ADAPTER_DIR_DRIVE}) over the repo's bundled copy...")
    shutil.copytree(ADAPTER_DIR_DRIVE, repo_adapter_dir, dirs_exist_ok=True)
else:
    print(f"No Drive adapter found — using the repo's bundled adapter at {repo_adapter_dir}.")

# Prefer Colab's Secrets manager (key icon in the left sidebar) so you never have
# to paste the key again — falls back to a hidden prompt if it isn't set there.
groq_key = None
try:
    from google.colab import userdata
    groq_key = userdata.get("GROQ_API_KEY")
    print("Using GROQ_API_KEY from Colab Secrets.")
except Exception:
    groq_key = None

if not groq_key:
    groq_key = getpass("GROQ_API_KEY not found in Colab Secrets — paste it here (input hidden, not printed): ")

with open(os.path.join(REPO_DIR, ".env"), "w", encoding="utf-8") as f:
    f.write(f"GROQ_API_KEY={groq_key}\n")
print(".env written inside the cloned repo.")
del groq_key

## Step 3 — Launch Streamlit in the background

Starts `streamlit run src/app.py` as a background process inside the Colab VM. Its own
startup (semantic-chunking corpus indexing, ~a couple of minutes) happens the first time
someone loads the page through the tunnel below, not in this cell.

In [ ]:
import subprocess, sys, time, os

log_path = "/content/streamlit_log.txt"
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", os.path.join(REPO_DIR, "src", "app.py"),
        "--server.port", str(STREAMLIT_PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
    cwd=REPO_DIR,
)

time.sleep(5)
if streamlit_proc.poll() is not None:
    print("Streamlit exited immediately — check the log:")
    print(open(log_path).read())
else:
    print(f"Streamlit running (pid {streamlit_proc.pid}) on port {STREAMLIT_PORT}.")
    print(f"Log: {log_path}")

## Step 4 — Tunnel the port out (ngrok)

Needs a free ngrok account: sign up at ngrok.com, then copy your authtoken from
the dashboard ("Your Authtoken" page). Read from Colab's **Secrets** manager
(add a secret named `NGROK_AUTHTOKEN` once) if present, otherwise you'll be
prompted for it (hidden input) below — same pattern as the Groq key.

Unlike the previous `localtunnel` setup, this cell finishes immediately —
the tunnel keeps running in the background as long as this Colab runtime
stays connected. Run the **Cleanup** cell below to actually tear it down.


In [ ]:
from pyngrok import ngrok
from getpass import getpass

ngrok_token = None
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
    print("Using NGROK_AUTHTOKEN from Colab Secrets.")
except Exception:
    ngrok_token = None

if not ngrok_token:
    ngrok_token = getpass("NGROK_AUTHTOKEN not found in Colab Secrets — paste it here (input hidden, not printed): ")

ngrok.set_auth_token(ngrok_token)
del ngrok_token

public_url = ngrok.connect(STREAMLIT_PORT, "http")
print(f"Demo is live at: {public_url}")
print("Run the Cleanup cell below when you're done to tear down the tunnel.")

## Cleanup (optional)

Run this if you want to stop Streamlit without interrupting/restarting the whole runtime.

In [ ]:
try:
    ngrok.kill()
    print("ngrok tunnel closed.")
except NameError:
    print("No ngrok tunnel found in this session.")

try:
    streamlit_proc.terminate()
    print("Streamlit process terminated.")
except NameError:
    print("No Streamlit process found in this session.")